# Incremental Medallion Pipeline with SCD Type 2

This version keeps the original `parent.csv` as the initial load and treats files named `parent_*.csv` as newly arriving customer changes.

- Bronze: append-only raw history using Auto Loader.
- Silver parent: SCD Type 2 customer dimension.
- Silver child: cleaned order data.
- Gold: current customer + order view.
- Auto Loader checkpointing prevents the same file path from being processed twice.


In [0]:
# ============================================================
# CONFIGURATION + AUTO LOADER
# ============================================================


landing_path = "abfss://landing@harekrishna.dfs.core.windows.net/"
schema_path = f"{landing_path}_schemas/parent"
checkpoint_path = f"{landing_path}_checkpoints/parent"

spark.sql("CREATE SCHEMA IF NOT EXISTS hk_db_workspace.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS hk_db_workspace.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS hk_db_workspace.gold")


# ============================================================
# BRONZE - AUTO LOADER
# ============================================================

# PySpark is used here because Auto Loader uses cloudFiles.
parent_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("header", "true")
    .option("pathGlobFilter", "parent_*.csv")
    .load(landing_path)
    .selectExpr(
        "*",
        "current_timestamp() AS _ingested_at",
        "_metadata.file_path AS _source_file"
    )
)

query = (
    parent_df.writeStream
    .option("checkpointLocation", checkpoint_path)
    #.trigger(availableNow=True)
    .trigger(processingTime="10 seconds")
    .toTable("hk_db_workspace.bronze.parent")
)

query.awaitTermination()


# ============================================================
# SILVER - LATEST PARENT DATA
# ============================================================

# Create latest incoming customer batch.
spark.sql("""
CREATE OR REPLACE TEMP VIEW incoming_parent AS

SELECT
    customer_id,
    customer_name,
    state

FROM hk_db_workspace.bronze.parent

WHERE _ingested_at = (
    SELECT MAX(_ingested_at)
    FROM hk_db_workspace.bronze.parent
)

AND customer_id IS NOT NULL
AND customer_name IS NOT NULL

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY customer_id
    ORDER BY _ingested_at DESC
) = 1
""")


# ============================================================
# SILVER - CREATE SCD2 TABLE
# ============================================================

spark.sql("""
CREATE TABLE IF NOT EXISTS hk_db_workspace.silver.parent
USING DELTA
AS

SELECT
    customer_id,
    customer_name,
    state,
    current_timestamp() AS effective_from,
    CAST(NULL AS TIMESTAMP) AS effective_to,
    TRUE AS is_current

FROM incoming_parent
""")


# ============================================================
# SILVER - FIND CHANGES
# ============================================================

spark.sql("""
CREATE OR REPLACE TEMP VIEW customer_changes AS

SELECT
    s.customer_id,
    s.customer_name,
    s.state

FROM incoming_parent s

LEFT JOIN hk_db_workspace.silver.parent t
    ON s.customer_id = t.customer_id
    AND t.is_current = TRUE

WHERE
       t.customer_id IS NULL
    OR NOT (s.customer_name <=> t.customer_name)
    OR NOT (s.state <=> t.state)
""")


# ============================================================
# SCD2 STEP 1 - EXPIRE OLD RECORD
# ============================================================

spark.sql("""
MERGE INTO hk_db_workspace.silver.parent t

USING customer_changes s

ON  t.customer_id = s.customer_id
AND t.is_current = TRUE

WHEN MATCHED THEN UPDATE SET
    t.effective_to = current_timestamp(),
    t.is_current = FALSE
""")


# ============================================================
# SCD2 STEP 2 - INSERT NEW VERSION
# ============================================================

spark.sql("""
INSERT INTO hk_db_workspace.silver.parent

SELECT
    customer_id,
    customer_name,
    state,
    current_timestamp() AS effective_from,
    CAST(NULL AS TIMESTAMP) AS effective_to,
    TRUE AS is_current

FROM customer_changes
""")


# ============================================================
# SILVER - CHILD / ORDERS
# ============================================================

# Read child CSV using Spark SQL.
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW raw_orders
USING CSV
OPTIONS (
    path '{landing_path}child.csv',
    header 'true',
    inferSchema 'true'
)
""")


# Clean and load orders.
spark.sql("""
CREATE OR REPLACE TABLE hk_db_workspace.silver.child
USING DELTA
AS

SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY order_id
               ORDER BY order_id
           ) AS rn

    FROM raw_orders

    WHERE order_id IS NOT NULL
      AND customer_id IS NOT NULL
)

WHERE rn = 1
""")


# ============================================================
# GOLD
# ============================================================

spark.sql("""
CREATE OR REPLACE TABLE hk_db_workspace.gold.customer_orders
USING DELTA
AS

SELECT
    o.*,
    c.customer_name,
    c.state

FROM hk_db_workspace.silver.child o

INNER JOIN hk_db_workspace.silver.parent c
    ON o.customer_id = c.customer_id

WHERE c.is_current = TRUE
""")


# ============================================================
# VERIFY
# ============================================================

display(spark.sql("""
SELECT *
FROM hk_db_workspace.bronze.parent
"""))

display(spark.sql("""
SELECT *
FROM hk_db_workspace.silver.parent
ORDER BY customer_id, effective_from
"""))

display(spark.sql("""
SELECT *
FROM hk_db_workspace.gold.customer_orders
"""))

In [0]:
%python
-- SELECT COUNT(*) AS total_rows
-- FROM hk_db_workspace.bronze.parent;


print(query.isActive)


In [0]:
# CI/CD deployment test
print("Testing automatic deployment")